In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="134_OG5odCDQUGuz6NND7LfkGmBcI5dpZ", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


# GPipe and 1F1B Schedules -- Vizuara

## What You Will Learn

In this notebook, you will:

1. **Implement the GPipe schedule** (all forwards, then all backwards) from scratch
2. **Implement the 1F1B schedule** (interleaved forward and backward passes)
3. **Compare bubble fractions** and verify they are identical for GPipe and 1F1B
4. **Track peak activation memory** and see the dramatic 1F1B advantage
5. **Visualize both schedules** side by side with timeline charts

**Estimated time**: 40-60 minutes

**Prerequisites**: Notebook 01 (pipeline bubble basics)

In [ ]:
#@title 🎧 Code Walkthrough: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

%matplotlib inline
np.random.seed(42)

print("Setup complete!")
print("In this notebook we build GPipe and 1F1B schedulers from scratch.")

In [ ]:
# Core data structures (same as Notebook 01)

@dataclass
class PipelineStage:
    stage_id: int
    forward_time: float
    backward_time: float

@dataclass
class ScheduleEvent:
    stage_id: int
    microbatch_id: int
    is_forward: bool
    start_time: float
    end_time: float

def create_uniform_pipeline(num_stages: int, forward_time: float = 100.0) -> List[PipelineStage]:
    return [
        PipelineStage(stage_id=i, forward_time=forward_time, backward_time=forward_time * 2.0)
        for i in range(num_stages)
    ]

def compute_bubble_fraction(events: List[ScheduleEvent], num_stages: int) -> dict:
    total_time = max(e.end_time for e in events)
    total_gpu_time = num_stages * total_time
    busy_per_stage = {s: 0.0 for s in range(num_stages)}
    for e in events:
        busy_per_stage[e.stage_id] += (e.end_time - e.start_time)
    total_busy = sum(busy_per_stage.values())
    total_idle = total_gpu_time - total_busy
    return {
        "total_time": total_time,
        "total_gpu_time": total_gpu_time,
        "total_busy": total_busy,
        "total_idle": total_idle,
        "bubble_fraction": total_idle / total_gpu_time,
        "busy_per_stage": busy_per_stage
    }

print("Core data structures loaded.")

In [ ]:
#@title 🎧 Listen: Gpipe Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_gpipe_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. Implementing the GPipe Schedule

GPipe (Huang et al., 2019) splits the mini-batch into $M$ micro-batches and runs the schedule in two phases:

1. **All forward passes**: Micro-batches flow through stages in a pipelined fashion.
2. **All backward passes**: After every micro-batch has completed its forward pass through all stages, backward passes run.

The key constraint: **no backward pass starts until all forward passes are done**.

In [ ]:
#@title 🎧 Code Walkthrough: Gpipe Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_gpipe_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def schedule_gpipe(stages: List[PipelineStage], num_microbatches: int) -> List[ScheduleEvent]:
    """GPipe schedule: all forwards first, then all backwards.

    Micro-batches are pipelined within each phase.
    """
    events = []
    P = len(stages)
    M = num_microbatches

    # Track when each stage becomes free
    stage_free_at = [0.0] * P

    # Track when each micro-batch's forward finishes at each stage
    fwd_done = {}  # (stage_id, mb_id) -> end_time

    # Phase 1: All forward passes
    for mb in range(M):
        for s in range(P):
            # Stage s can start when:
            # (a) it is free from previous work
            # (b) the previous stage finished this micro-batch's forward
            earliest = stage_free_at[s]
            if s > 0:
                earliest = max(earliest, fwd_done[(s - 1, mb)])

            start = earliest
            end = start + stages[s].forward_time

            events.append(ScheduleEvent(
                stage_id=s, microbatch_id=mb,
                is_forward=True, start_time=start, end_time=end
            ))
            stage_free_at[s] = end
            fwd_done[(s, mb)] = end

    # Phase 2: All backward passes (reversed micro-batch order)
    bwd_done = {}
    for mb in range(M - 1, -1, -1):
        for s in range(P - 1, -1, -1):
            earliest = stage_free_at[s]
            if s < P - 1:
                earliest = max(earliest, bwd_done[(s + 1, mb)])

            start = earliest
            end = start + stages[s].backward_time

            events.append(ScheduleEvent(
                stage_id=s, microbatch_id=mb,
                is_forward=False, start_time=start, end_time=end
            ))
            stage_free_at[s] = end
            bwd_done[(s, mb)] = end

    return events

# Test GPipe
P = 4
M = 8
stages = create_uniform_pipeline(P, forward_time=100.0)
events_gpipe = schedule_gpipe(stages, M)
stats_gpipe = compute_bubble_fraction(events_gpipe, P)

print(f"GPipe Schedule (P={P}, M={M}):")
print(f"  Total time: {stats_gpipe['total_time']:.0f} ms")
print(f"  Bubble fraction: {stats_gpipe['bubble_fraction']:.1%}")


In [ ]:
#@title 🎧 Listen: Gpipe Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_gpipe_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print(f"  Theoretical: {(P-1)/(P-1+M):.1%}")

In [ ]:
#@title 🎧 Listen: 1f1b Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_1f1b_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Implementing the 1F1B Schedule

1F1B (One Forward, One Backward) from PipeDream (Narayanan et al., 2019) has three phases:

1. **Warmup**: Stage $k$ runs $P - 1 - k$ forward passes to fill the pipeline.
2. **Steady state**: Each stage alternates -- one forward, then one backward.
3. **Cooldown**: Remaining backward passes drain the pipeline.

The bubble fraction is **identical** to GPipe, but the memory usage is dramatically better.

In [ ]:
#@title 🎧 Code Walkthrough: 1f1b Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_1f1b_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def schedule_1f1b(stages: List[PipelineStage], num_microbatches: int) -> List[ScheduleEvent]:
    """1F1B schedule: interleave forward and backward passes.

    Phase 1: Warmup (fill pipeline with forwards)
    Phase 2: Steady state (alternate 1 forward + 1 backward)
    Phase 3: Cooldown (drain remaining backwards)
    """
    events = []
    P = len(stages)
    M = num_microbatches

    # Track when each stage is free
    stage_free_at = [0.0] * P

    # Track forward completion times for dependency
    fwd_done = {}  # (stage_id, mb_id) -> end_time
    bwd_done = {}  # (stage_id, mb_id) -> end_time

    # For each stage, determine schedule
    for s in range(P):
        num_warmup = P - 1 - s
        num_steady = M - num_warmup

        fwd_idx = 0  # which micro-batch to forward next
        bwd_idx = 0  # which micro-batch to backward next

        # Phase 1: Warmup -- run num_warmup forward passes
        for i in range(num_warmup):
            mb = fwd_idx
            earliest = stage_free_at[s]
            if s > 0:
                earliest = max(earliest, fwd_done[(s - 1, mb)])

            start = earliest
            end = start + stages[s].forward_time
            events.append(ScheduleEvent(
                stage_id=s, microbatch_id=mb,
                is_forward=True, start_time=start, end_time=end
            ))
            stage_free_at[s] = end
            fwd_done[(s, mb)] = end
            fwd_idx += 1

        # Phase 2: Steady state -- alternate 1 forward + 1 backward
        for i in range(num_steady):
            # Forward pass
            mb_fwd = fwd_idx
            earliest_fwd = stage_free_at[s]
            if s > 0:
                earliest_fwd = max(earliest_fwd, fwd_done[(s - 1, mb_fwd)])

            start_fwd = earliest_fwd
            end_fwd = start_fwd + stages[s].forward_time
            events.append(ScheduleEvent(
                stage_id=s, microbatch_id=mb_fwd,
                is_forward=True, start_time=start_fwd, end_time=end_fwd
            ))
            stage_free_at[s] = end_fwd
            fwd_done[(s, mb_fwd)] = end_fwd
            fwd_idx += 1

            # Backward pass
            mb_bwd = bwd_idx
            earliest_bwd = stage_free_at[s]
            if s < P - 1:
                earliest_bwd = max(earliest_bwd, bwd_done[(s + 1, mb_bwd)])

            start_bwd = earliest_bwd
            end_bwd = start_bwd + stages[s].backward_time
            events.append(ScheduleEvent(
                stage_id=s, microbatch_id=mb_bwd,
                is_forward=False, start_time=start_bwd, end_time=end_bwd
            ))
            stage_free_at[s] = end_bwd
            bwd_done[(s, mb_bwd)] = end_bwd
            bwd_idx += 1

        # Phase 3: Cooldown -- run remaining backward passes
        for i in range(num_warmup):
            mb = bwd_idx
            earliest = stage_free_at[s]
            if s < P - 1:
                earliest = max(earliest, bwd_done[(s + 1, mb)])

            start = earliest
            end = start + stages[s].backward_time
            events.append(ScheduleEvent(
                stage_id=s, microbatch_id=mb,
                is_forward=False, start_time=start, end_time=end
            ))
            stage_free_at[s] = end
            bwd_done[(s, mb)] = end
            bwd_idx += 1

    return events

# Test 1F1B
events_1f1b = schedule_1f1b(stages, M)
stats_1f1b = compute_bubble_fraction(events_1f1b, P)

print(f"1F1B Schedule (P={P}, M={M}):")
print(f"  Total time: {stats_1f1b['total_time']:.0f} ms")
print(f"  Bubble fraction: {stats_1f1b['bubble_fraction']:.1%}")


In [ ]:
#@title 🎧 Listen: 1f1b Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_1f1b_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print(f"  Theoretical: {(P-1)/(P-1+M):.1%}")

In [ ]:
#@title 🎧 Transition: Viz Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_viz_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Visualizing Both Schedules Side by Side

Let us see the GPipe and 1F1B schedules visually to understand the structural difference.

In [ ]:
#@title 🎧 What to Look For: Viz Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_viz_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def plot_pipeline_timeline(events: List[ScheduleEvent], num_stages: int, title: str,
                           figsize=(18, 5), ax=None):
    """Visualize a pipeline schedule as a Gantt chart."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    total_time = max(e.end_time for e in events)

    fwd_color = '#4A90D9'
    bwd_color = '#E8634F'
    idle_color = '#E8E8E8'

    for s in range(num_stages):
        ax.barh(s, total_time, left=0, height=0.7, color=idle_color, edgecolor='white', linewidth=0.5)

    for e in events:
        color = fwd_color if e.is_forward else bwd_color
        duration = e.end_time - e.start_time
        ax.barh(e.stage_id, duration, left=e.start_time, height=0.7,
                color=color, edgecolor='white', linewidth=0.8)
        label = f"F{e.microbatch_id}" if e.is_forward else f"B{e.microbatch_id}"
        if duration > 40:
            ax.text(e.start_time + duration / 2, e.stage_id, label,
                    ha='center', va='center', fontsize=7, fontweight='bold', color='white')

    ax.set_yticks(range(num_stages))
    ax.set_yticklabels([f'GPU {i}' for i in range(num_stages)])
    ax.set_xlabel('Time (ms)', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.invert_yaxis()

    stats = compute_bubble_fraction(events, num_stages)
    ax.text(0.02, 0.02, f"Bubble: {stats['bubble_fraction']:.1%}",
            transform=ax.transAxes, fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Plot both side by side
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 8))

plot_pipeline_timeline(events_gpipe, P, f"GPipe (P={P}, M={M})", ax=ax1)
plot_pipeline_timeline(events_1f1b, P, f"1F1B (P={P}, M={M})", ax=ax2)

# Add legend to the figure
legend_elements = [
    mpatches.Patch(facecolor='#4A90D9', label='Forward'),
    mpatches.Patch(facecolor='#E8634F', label='Backward'),
    mpatches.Patch(facecolor='#E8E8E8', label='Idle'),
]
fig.legend(handles=legend_elements, loc='upper right', fontsize=11, ncol=3,
           bbox_to_anchor=(0.98, 1.0))

plt.tight_layout()
plt.show()

print(f"GPipe bubble:  {stats_gpipe['bubble_fraction']:.1%}")
print(f"1F1B bubble:   {stats_1f1b['bubble_fraction']:.1%}")


In [ ]:
#@title 🎧 Listen: Viz Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_viz_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("\nBoth have the same bubble fraction!")
print("The difference is in MEMORY, not in throughput.")

In [ ]:
#@title 🎧 Transition: Memory Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_memory_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Tracking Peak Activation Memory

This is where the real difference between GPipe and 1F1B shows up.

- **GPipe**: All forwards complete before any backward starts. Each stage must store activations for **all $M$** micro-batches.
- **1F1B**: Backward passes start early. Each stage stores activations for at most **$P$** micro-batches.

Let us track and compare the peak activation memory.

In [ ]:
#@title 🎧 Code Walkthrough: Memory Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_memory_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def track_activation_memory(events: List[ScheduleEvent], num_stages: int) -> dict:
    """Track peak activation memory per stage over time.

    Activations are stored after a forward pass and freed after the
    corresponding backward pass at the same stage.
    """
    # Sort events by start time
    sorted_events = sorted(events, key=lambda e: (e.start_time, not e.is_forward))

    # Track stored activations per stage
    stored = {s: set() for s in range(num_stages)}  # which micro-batches are stored
    peak_count = {s: 0 for s in range(num_stages)}

    # Timeline of memory usage per stage
    memory_timeline = {s: [] for s in range(num_stages)}  # (time, count)

    for e in sorted_events:
        s = e.stage_id
        if e.is_forward:
            # Forward completes: store activation
            stored[s].add(e.microbatch_id)
            count = len(stored[s])
            peak_count[s] = max(peak_count[s], count)
            memory_timeline[s].append((e.end_time, count))
        else:
            # Backward completes: free activation
            stored[s].discard(e.microbatch_id)
            count = len(stored[s])
            memory_timeline[s].append((e.end_time, count))

    return {
        "peak_per_stage": peak_count,
        "max_peak": max(peak_count.values()),
        "timeline": memory_timeline
    }

# Compare memory
mem_gpipe = track_activation_memory(events_gpipe, P)
mem_1f1b = track_activation_memory(events_1f1b, P)

print(f"Peak activations stored per stage:")
print(f"{'Stage':>6} {'GPipe':>8} {'1F1B':>8}")
print("-" * 28)
for s in range(P):
    print(f"{s:>6} {mem_gpipe['peak_per_stage'][s]:>8} {mem_1f1b['peak_per_stage'][s]:>8}")
print(f"{'MAX':>6} {mem_gpipe['max_peak']:>8} {mem_1f1b['max_peak']:>8}")
print()
print(f"GPipe stores up to M={M} activations per stage.")
print(f"1F1B stores up to P={P} activations per stage.")
print(f"Memory reduction: {M / P:.0f}x fewer stored activations!")

In [ ]:
#@title 🎧 What to Look For: Memory Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_memory_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize memory over time for one stage
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for s_plot, ax, schedule_name, mem_data in [
    (0, ax1, "GPipe", mem_gpipe),
    (0, ax2, "1F1B", mem_1f1b)
]:
    timeline = mem_data["timeline"][s_plot]
    times = [t for t, c in timeline]
    counts = [c for t, c in timeline]

    ax.step(times, counts, where='post', linewidth=2, color='#4A90D9')
    ax.fill_between(times, counts, step='post', alpha=0.3, color='#4A90D9')
    ax.axhline(y=mem_data['peak_per_stage'][s_plot], color='red', linestyle='--',
               label=f'Peak = {mem_data["peak_per_stage"][s_plot]}')
    ax.set_xlabel('Time (ms)', fontsize=11)
    ax.set_ylabel('Stored Activations', fontsize=11)
    ax.set_title(f'{schedule_name} -- Stage 0 Memory', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_ylim(0, M + 1)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("GPipe: Memory ramps up to M during the forward phase, then drops during backward.")


In [ ]:
#@title 🎧 Listen: Memory Viz Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_memory_viz_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("1F1B:  Memory peaks at P and stays bounded because backwards start early.")

In [ ]:
#@title 🎧 Transition: Scaling Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_scaling_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Memory Scaling: The Decisive Advantage

Let us see how memory scales as we increase $M$ for both schedules.

In [ ]:
#@title 🎧 Code Walkthrough: Scaling Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_scaling_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
P = 4
M_values = [4, 8, 16, 32, 64]
activation_size_mb = 50  # MB per micro-batch per stage

gpipe_memory = []
f1b_memory = []
gpipe_bubble = []
f1b_bubble = []

for M in M_values:
    stages = create_uniform_pipeline(P, forward_time=100.0)

    events_g = schedule_gpipe(stages, M)
    events_f = schedule_1f1b(stages, M)

    mem_g = track_activation_memory(events_g, P)
    mem_f = track_activation_memory(events_f, P)

    gpipe_memory.append(mem_g['max_peak'] * activation_size_mb)
    f1b_memory.append(mem_f['max_peak'] * activation_size_mb)

    stats_g = compute_bubble_fraction(events_g, P)
    stats_f = compute_bubble_fraction(events_f, P)
    gpipe_bubble.append(stats_g['bubble_fraction'])
    f1b_bubble.append(stats_f['bubble_fraction'])

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Memory comparison
x = np.arange(len(M_values))
width = 0.35
bars1 = ax1.bar(x - width/2, gpipe_memory, width, label='GPipe', color='#E8634F')
bars2 = ax1.bar(x + width/2, f1b_memory, width, label='1F1B', color='#4A90D9')

ax1.set_xlabel('Number of Micro-Batches (M)', fontsize=11)
ax1.set_ylabel('Peak Activation Memory (MB)', fontsize=11)
ax1.set_title('Peak Memory: GPipe vs 1F1B', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(M_values)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{bar.get_height():.0f}', ha='center', fontsize=9)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{bar.get_height():.0f}', ha='center', fontsize=9)

# Bubble comparison
ax2.plot(M_values, [b * 100 for b in gpipe_bubble], 'o-', color='#E8634F',
         linewidth=2, markersize=8, label='GPipe')
ax2.plot(M_values, [b * 100 for b in f1b_bubble], 's--', color='#4A90D9',
         linewidth=2, markersize=8, label='1F1B')
ax2.set_xlabel('Number of Micro-Batches (M)', fontsize=11)
ax2.set_ylabel('Bubble Fraction (%)', fontsize=11)
ax2.set_title('Bubble: GPipe vs 1F1B (identical!)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight:")
print("- GPipe and 1F1B have the SAME bubble fraction.")
print("- But GPipe memory grows linearly with M (bad).")
print("- 1F1B memory stays constant at P (good).")
print(f"- At M=64: GPipe uses {gpipe_memory[-1]:.0f}MB, 1F1B uses {f1b_memory[-1]:.0f}MB.")


In [ ]:
#@title 🎧 Listen: Scaling Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_scaling_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print(f"  That is a {gpipe_memory[-1] / f1b_memory[-1]:.0f}x reduction!")

In [ ]:
#@title 🎧 Transition: Pseudocode Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_pseudocode_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. 1F1B Pseudocode Deep Dive

Let us trace through the 1F1B algorithm step-by-step for one stage to really understand the three phases.

In [ ]:
#@title 🎧 Code Walkthrough: Pseudocode Stage0
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_pseudocode_stage0.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def trace_1f1b_stage(stage_id: int, num_stages: int, num_microbatches: int):
    """Print the 1F1B schedule for a single stage."""
    P = num_stages
    M = num_microbatches
    k = stage_id

    num_warmup = P - 1 - k
    num_steady = M - num_warmup

    print(f"\n{'='*50}")
    print(f"Stage {k} (of {P}), M={M} micro-batches")
    print(f"{'='*50}")

    fwd_idx = 0
    bwd_idx = 0
    stored = []  # track stored activations

    # Phase 1: Warmup
    print(f"\nPhase 1: WARMUP ({num_warmup} forward passes)")
    for i in range(num_warmup):
        stored.append(fwd_idx)
        print(f"  Forward(MB={fwd_idx})  -> stored: {stored} ({len(stored)} activations)")
        fwd_idx += 1

    # Phase 2: Steady state
    print(f"\nPhase 2: STEADY STATE ({num_steady} iterations of 1F+1B)")
    for i in range(num_steady):
        stored.append(fwd_idx)
        print(f"  Forward(MB={fwd_idx})  -> stored: {stored} ({len(stored)} activations)")
        fwd_idx += 1

        stored.remove(bwd_idx)
        print(f"  Backward(MB={bwd_idx}) -> stored: {stored} ({len(stored)} activations)")
        bwd_idx += 1

    # Phase 3: Cooldown
    print(f"\nPhase 3: COOLDOWN ({num_warmup} backward passes)")
    for i in range(num_warmup):
        stored.remove(bwd_idx)
        print(f"  Backward(MB={bwd_idx}) -> stored: {stored} ({len(stored)} activations)")
        bwd_idx += 1

    print(f"\nPeak activations stored: {P - 1 if k == 0 else max(P - 1 - k + 1, P - 1 - k)}")

# Trace stage 0 with P=4, M=8
trace_1f1b_stage(stage_id=0, num_stages=4, num_microbatches=8)

In [ ]:
#@title 🎧 Code Walkthrough: Pseudocode Stage3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_pseudocode_stage3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Trace stage 3 (last stage) -- it has 0 warmup
trace_1f1b_stage(stage_id=3, num_stages=4, num_microbatches=8)

In [ ]:
#@title 🎧 Listen: Pseudocode Explanation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_pseudocode_explanation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


Notice:
- Stage 0 needs $P-1 = 3$ warmup forwards before any backward can start.
- Stage 3 (the last stage) needs 0 warmup forwards -- it immediately enters steady state.
- During steady state, one activation is created (forward) and one is freed (backward) per iteration, keeping the count stable.

In [ ]:
#@title 🎧 Transition: Concrete Memory Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_concrete_memory_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Concrete Memory Comparison

Let us compute actual memory numbers for a realistic model.

In [ ]:
#@title 🎧 Code Walkthrough: Concrete Memory Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_24_concrete_memory_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compute_activation_memory(batch_size: int, seq_len: int, hidden_dim: int,
                               num_layers_per_stage: int, dtype_bytes: int = 2) -> float:
    """Approximate activation memory per micro-batch per stage in MB.

    Each transformer layer stores ~34*b*s*h bytes of activations (empirical).
    Simplified: we use seq_len * hidden_dim * batch_size * num_layers * factor.
    """
    # Each layer: input activations + intermediate (FFN) + attention
    # Rough estimate: ~12 * b * s * h bytes per layer for FP16
    bytes_per_layer = 12 * batch_size * seq_len * hidden_dim * dtype_bytes
    total_bytes = bytes_per_layer * num_layers_per_stage
    return total_bytes / (1024 ** 2)  # MB

# LLaMA 7B-like config
L = 32   # total layers
P = 4    # pipeline stages
h = 4096 # hidden dim
s = 4096 # sequence length
b = 1    # micro-batch size
layers_per_stage = L // P

act_per_mb = compute_activation_memory(b, s, h, layers_per_stage)

print(f"Model config: {L} layers, hidden={h}, seq_len={s}, micro-batch={b}")
print(f"Pipeline: {P} stages, {layers_per_stage} layers per stage")
print(f"Activation memory per micro-batch per stage: {act_per_mb:.0f} MB")
print()

print(f"{'M':>4} {'GPipe Peak (MB)':>16} {'1F1B Peak (MB)':>16} {'Reduction':>10}")
print("-" * 52)
for M in [4, 8, 16, 32, 64, 128]:
    gpipe_peak = M * act_per_mb
    f1b_peak = P * act_per_mb
    reduction = gpipe_peak / f1b_peak
    print(f"{M:>4} {gpipe_peak:>16,.0f} {f1b_peak:>16,.0f} {reduction:>9.0f}x")

print(f"\nWith 1F1B, you can increase M freely to reduce the bubble")


In [ ]:
#@title 🎧 Listen: Concrete Memory Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_25_concrete_memory_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print(f"without worrying about activation memory blowing up!")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_26_todo_1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Your Turn

### Exercise 1: Verify the Bubble Formula

Run both schedules for several $(P, M)$ configurations and verify that the bubble fraction matches $\frac{P-1}{P-1+M}$.

In [ ]:
#@title 🎧 Before You Start: Todo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_27_todo_1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Complete the verification table.
# For each config, run both schedulers and compare against the formula.

configs = [
    (2, 4), (2, 16), (4, 8), (4, 32), (8, 16), (8, 64)
]

print(f"{'P':>4} {'M':>4} {'Formula':>10} {'GPipe':>10} {'1F1B':>10} {'Match':>8}")
print("-" * 52)

for P, M in configs:
    stages = create_uniform_pipeline(P, forward_time=100.0)

    # TODO: Compute theoretical bubble
    theoretical = None  # REPLACE

    # TODO: Run GPipe and compute bubble
    # events_g = schedule_gpipe(stages, M)
    # stats_g = compute_bubble_fraction(events_g, P)
    gpipe_bubble = None  # REPLACE

    # TODO: Run 1F1B and compute bubble
    # events_f = schedule_1f1b(stages, M)
    # stats_f = compute_bubble_fraction(events_f, P)
    f1b_bubble = None  # REPLACE

    match = "---"  # TODO: check if all three are approximately equal


In [ ]:
#@title 🎧 Listen: Todo After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_28_todo_1_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
    print(f"{P:>4} {M:>4} {theoretical:>10} {gpipe_bubble:>10} {f1b_bubble:>10} {match:>8}")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_29_todo_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 2: Plot Memory Timeline for All Stages

Create a 4-panel figure showing the activation memory timeline for each stage (0-3) under the 1F1B schedule with P=4, M=16.

In [ ]:
#@title 🎧 Before You Start: Todo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_30_todo_2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Generate events and track memory for P=4, M=16
P = 4
M = 16
stages = create_uniform_pipeline(P, forward_time=100.0)

# TODO: Run 1F1B schedule
# events = schedule_1f1b(stages, M)
# mem = track_activation_memory(events, P)

# TODO: Create a 2x2 subplot figure
# For each stage (0-3), plot the memory timeline
# Use step plots (ax.step) with fill_between
# Mark the peak with a horizontal dashed line
# Add a title showing the peak count for each stage

# fig, axes = plt.subplots(2, 2, figsize=(14, 8))
# for s in range(P):
#     ax = axes[s // 2][s % 2]
#     timeline = mem['timeline'][s]
#     ...


In [ ]:
#@title 🎧 Listen: Todo After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_31_todo_2_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("TODO: Implement the 4-panel memory timeline visualization.")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_32_todo_3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 3: When Does 1F1B Start Paying Off?

For $P = 8$ stages, at what value of $M$ does GPipe require more than 10 GB of activation memory while 1F1B stays under 2 GB? Assume 200 MB per micro-batch per stage.

In [ ]:
#@title 🎧 Before You Start: Todo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_33_todo_3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Find the crossover point
#
# GPipe peak memory = M * 200 MB
# 1F1B peak memory = P * 200 MB = 8 * 200 = 1600 MB (always under 2 GB)
#
# Find smallest M where GPipe exceeds 10 GB = 10,000 MB
# M * 200 > 10,000 => M > ???

P = 8
act_size = 200  # MB per micro-batch per stage

# TODO: compute the answer
# What is the bubble fraction at this M?
# Is the bubble acceptable?

print("TODO: Find the M where GPipe exceeds 10 GB.")
print("Compute the bubble fraction at that M.")


In [ ]:
#@title 🎧 Listen: Todo After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_34_todo_3_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("Explain why 1F1B is essential for large-scale training.")

In [ ]:
#@title 🎧 Transition: Summary Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_35_summary_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Summary


In [ ]:
#@title 🎧 Listen: Summary Table
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_36_summary_table.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


| Property | GPipe | 1F1B |
|---|---|---|
| Bubble fraction | $(P-1)/(P-1+M)$ | $(P-1)/(P-1+M)$ |
| Peak activations | $M$ per stage | $P$ per stage |
| Memory scaling | Grows with $M$ | Independent of $M$ |
| Complexity | Simpler | Slightly more complex |

**Key takeaways:**

1. GPipe and 1F1B have **identical bubble fractions**.
2. 1F1B is strictly better because it **decouples bubble size from memory usage**.
3. With 1F1B, you can freely increase $M$ to shrink the bubble without memory blowup.
4. This is why **all modern pipeline implementations** use 1F1B or its variants.

**Next notebook**: We will explore **Interleaved 1F1B**, which reduces the bubble by a further factor of $v$ using virtual pipeline stages.

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_37_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("=" * 60)
print("SUMMARY: GPipe vs 1F1B")
print("=" * 60)
print()
print("Both schedules achieve the same bubble fraction.")
print("The critical difference is MEMORY:")
print(f"  GPipe: stores M={M} activations per stage")
print(f"  1F1B:  stores P={P} activations per stage")
print()
print("1F1B strictly dominates GPipe for training.")
print("GPipe is historically important but no longer used in practice.")
print()
print("Next: Interleaved 1F1B with virtual stages!")